# Retrosynthesis route search — synthesizability benchmark

Runs AiZynthFinder (USPTO expansion policy, ZINC stock) on 175 molecules drawn from
seven reference sets, to ask what the fast scores cannot: **does a real route search
agree with them?**

**Why the extra setup below:** Colab now ships Python 3.13, and AiZynthFinder supports
up to 3.12. So we build a private Python 3.11 environment inside the notebook with `uv`
and call AiZynthFinder's command-line tool from it. The notebook's own kernel stays on
3.13 and is only used for bookkeeping — nothing needs to be imported into it.

Use a **CPU runtime** (Runtime → Change runtime type → CPU). No GPU is needed and it
would only consume quota.

Expect 2–3 hours. Results are appended to Google Drive after every batch of 10, so a
disconnect loses at most one batch — just re-run the search cell and it resumes.


## 1. Build a Python 3.11 environment and install AiZynthFinder

In [ ]:
!pip -q install uv
!uv venv /content/aiz --python 3.11 -q --clear
!uv pip install -q --python /content/aiz/bin/python aizynthfinder
# aizynthfinder pins numpy<2, but dask pulls a pyarrow built against numpy 2,
# which then refuses to import. Hold pyarrow at the last numpy-1.x-compatible line.
!uv pip install -q --python /content/aiz/bin/python "pyarrow<19"
!/content/aiz/bin/python -c "import sys, importlib.metadata as m; import aizynthfinder; print('aizynthfinder', m.version('aizynthfinder'), '| python', sys.version.split()[0])"


## 2. Download the published models

`download_public_data` fetches the USPTO expansion policy, the filter policy and the
ZINC stock from Zenodo (about 1 GB) and writes a matching `config.yml`.


In [ ]:
!mkdir -p /content/aizynth_data
!cd /content/aizynth_data && /content/aiz/bin/download_public_data .
!ls -la /content/aizynth_data


## 3. Mount Drive and load the input

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, pandas as pd, multiprocessing
DRIVE_DIR = '/content/drive/MyDrive/synthscore'
assert os.path.isdir(DRIVE_DIR), f'folder not found: {DRIVE_DIR} — check the name in My Drive'
IN_CSV  = os.path.join(DRIVE_DIR, 'route_search_input.csv')
OUT_CSV = os.path.join(DRIVE_DIR, 'route_search_results.csv')

df = pd.read_csv(IN_CSV)
NPROC = max(1, multiprocessing.cpu_count())
print(df.shape, '| cpus:', NPROC)
print(df.cell.value_counts().to_string())


## 4. Run the route search

Molecules go through `aizynthcli` in batches of 10. For each one AiZynthFinder reports
whether it found a complete route to purchasable starting materials (`is_solved`) and how
long that route is (`number_of_steps`).

One thing to watch: the ZINC set is drawn from the *same* stock AiZynthFinder searches
against, so those molecules are already purchasable and come back solved in **zero**
steps. That is not a route success — it is the search recognising its own catalogue.
`n_steps == 0` marks these, and they are counted separately in the summary.


In [ ]:
import os, glob, subprocess
# preflight — confirm the pieces from cells 1 and 2 are actually where we expect
AIZ  = '/content/aiz/bin/aizynthcli'
CFG  = '/content/aizynth_data/config.yml'
print('aizynthcli present :', os.path.exists(AIZ))
print('config.yml present :', os.path.exists(CFG))
print('data dir contents  :', sorted(os.path.basename(p) for p in glob.glob('/content/aizynth_data/*'))[:12])
if os.path.exists(CFG):
    print('--- config.yml ---'); print(open(CFG).read()[:600])
if os.path.exists(AIZ):
    v = subprocess.run([AIZ, '--help'], capture_output=True, text=True)
    print('aizynthcli --help rc:', v.returncode)
    if v.returncode != 0: print(v.stderr[-600:])


In [ ]:
import subprocess, time, pandas as pd, os

BATCH   = 10
AIZ_BIN = '/content/aiz/bin'
CFG     = '/content/aizynth_data/config.yml'
KEEP = ['target','is_solved','number_of_steps','number_of_nodes','top_score',
        'number_of_routes','number_of_solved_routes','search_time',
        'number_of_precursors','number_of_precursors_in_stock']

# With --nproc, aizynthcli spawns child processes by invoking a bare `aizynthcli`,
# which it looks up on PATH. We call it by absolute path, so the venv's bin directory
# has to be added to PATH or those children cannot be found.
ENV = dict(os.environ, PATH=AIZ_BIN + ':' + os.environ.get('PATH', ''))

def run_batch(smi_path, out_path, nproc):
    cmd = [f'{AIZ_BIN}/aizynthcli', '--config', CFG,
           '--smiles', smi_path, '--output', out_path]
    if nproc > 1:
        cmd += ['--nproc', str(nproc)]
    return subprocess.run(cmd, capture_output=True, text=True, env=ENV)

done = set()
if os.path.exists(OUT_CSV):
    done = set(pd.read_csv(OUT_CSV).route_id)
    print('resuming - already done:', len(done))

todo = df[~df.route_id.isin(done)].reset_index(drop=True)
print('to run:', len(todo), '| nproc:', NPROC)
t_start = time.time()

for b0 in range(0, len(todo), BATCH):
    chunk    = todo.iloc[b0:b0+BATCH]
    smi_path = f'/content/batch_{b0}.smi'
    out_path = f'/content/batch_{b0}.json.gz'
    with open(smi_path, 'w') as fh:
        fh.write('\n'.join(chunk.smiles.astype(str)) + '\n')

    p = run_batch(smi_path, out_path, NPROC)
    if p.returncode != 0 and NPROC > 1:
        print('parallel run failed; retrying this batch single-process')
        p = run_batch(smi_path, out_path, 1)
        if p.returncode == 0:
            NPROC = 1        # stay single-process for the remaining batches
    if p.returncode != 0:
        print('=== BATCH FAILED, exit', p.returncode, '===')
        print('--- stderr ---'); print(p.stderr[-2500:])
        print('--- stdout ---'); print(p.stdout[-1200:])
        raise SystemExit('route search failed - paste the output above')

    res = pd.read_json(out_path, orient='table')
    res = res[[c for c in KEEP if c in res.columns]].reset_index(drop=True)
    res.insert(0, 'route_id',     chunk.route_id.values)
    res.insert(1, 'cell',         chunk.cell.values)
    res.insert(2, 'known_status', chunk.known_status.values)

    res.to_csv(OUT_CSV, mode='a', header=not os.path.exists(OUT_CSV), index=False)
    n_done = len(pd.read_csv(OUT_CSV))
    print(f'{n_done:>4}/{len(df)}   elapsed {(time.time()-t_start)/60:.0f} min', flush=True)

print('\nDONE ->', OUT_CSV)


## 5. Summary

In [ ]:
import os, pandas as pd
if not os.path.exists(OUT_CSV):
    print('no results yet — the search cell above has not written anything')
else:
    res = pd.read_csv(OUT_CSV)
    res['trivial'] = res.number_of_steps.fillna(-1).eq(0)
    summ = res.groupby('cell').agg(
        n=('route_id','size'),
        solved_pct=('is_solved', lambda s: round(100*s.fillna(False).mean(), 1)),
        trivially_in_stock=('trivial','sum'),
        median_steps=('number_of_steps','median'),
        median_time_s=('search_time','median'))
    print(summ.to_string())
